In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import cv2
import joblib
import numpy as np
import pandas as pd
import onnxruntime as ort
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# ============================================================
# 2. CONFIGURATION
# ============================================================
DATA_PATH = "train_data.csv"
MOBILEVIT_ONNX_PATH = "mobilevit.onnx"
RESNET_ONNX_PATH = "Resnet.onnx"
RANDOM_FOREST_PATH = "best_random_forest_model.pkl"

IMG_SHAPE = (28, 28)
FEATURE_SIZE = 784

CLASS_NAMES = [
    "BENIGN",
    "DoS",
    "Fuzzy",
    "RPM",
    "Advanced",
    "Patator"
]

# ============================================================
# 3. CHECK FILES
# ============================================================
required_files = [
    DATA_PATH,
    MOBILEVIT_ONNX_PATH,
    RESNET_ONNX_PATH
]

for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Missing required file: {file}")

print("All base required files found.")

# ============================================================
# 4. LOAD DATASET
# ============================================================
print("\nLoading dataset...")
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

# Remove NaN and infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dataset shape: {df.shape}")

# ============================================================
# 5. LABEL MAPPING
# ============================================================
def map_label(label):
    if label == "BENIGN":
        return "BENIGN"
    elif "DoS" in label:
        return "DoS"
    elif "PortScan" in label or "Brute Force" in label:
        return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label:
        return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label:
        return "Advanced"
    elif "Patator" in label:
        return "Patator"
    else:
        return "Other"

df["Category"] = df["Label"].apply(map_label)
df = df[df["Category"] != "Other"]

print("\nClass Distribution:")
print(df["Category"].value_counts())

# ============================================================
# 6. FEATURE EXTRACTION
# ============================================================
print("\nPreparing features...")
features = df.select_dtypes(include=[np.number])
labels = df["Category"]

# ============================================================
# 7. FEATURE SCALING & SMART SUBSAMPLING
# ============================================================
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)
features_scaled = (features_scaled * 255).astype(np.uint8)

TARGET_SAMPLES = 3000
MIN_PER_CLASS = 15  

print(f"\nSubsampling dataset to {TARGET_SAMPLES} rows...")
sub_indices = []

for category in df["Category"].unique():
    category_indices = df[df["Category"] == category].index.tolist()
    n_take = min(len(category_indices), MIN_PER_CLASS)
    chosen = pd.Series(category_indices).sample(n=n_take, random_state=42).tolist()
    sub_indices.extend(chosen)

remaining_quota = TARGET_SAMPLES - len(sub_indices)

if remaining_quota > 0:
    remaining_indices = df.index.difference(sub_indices)
    remaining_df = df.loc[remaining_indices]
    
    rem_counts = remaining_df["Category"].value_counts()
    valid_categories = rem_counts[rem_counts >= 2].index
    strat_df = remaining_df[remaining_df["Category"].isin(valid_categories)]
    
    extra_indices, _ = train_test_split(
        strat_df.index.to_numpy(),
        train_size=remaining_quota,
        stratify=strat_df["Category"],
        random_state=42
    )
    sub_indices.extend(extra_indices)

# Sync arrays and tracking DataFrames
features_scaled = features_scaled[sub_indices]
features = features.iloc[sub_indices].reset_index(drop=True)
labels = df["Category"].iloc[sub_indices].reset_index(drop=True)

print(f"Final features shape for pipeline: {features_scaled.shape}")

# ============================================================
# 8. CREATE IMAGE FUNCTION
# ============================================================
def create_image_from_row(row, size=FEATURE_SIZE):
    data = np.array(row)
    if len(data) < size:
        data = np.pad(data, (0, size - len(data)), mode='constant')
    else:
        data = data[:size]
    img = data.reshape(IMG_SHAPE)
    return img.astype(np.uint8)

# ============================================================
# 9. IMAGE PREPROCESSING FOR ONNX
# ============================================================
def preprocess_resnet(img):
    img = cv2.resize(img, (224, 224))
    img = img.astype(np.float32) / 255.0
    img = np.stack([img] * 3, axis=-1)
    img = np.expand_dims(img, axis=0)
    return img.astype(np.float32)

def preprocess_mobilevit(img):
    img = img.astype(np.float32) / 255.0
    img = np.stack([img] * 3, axis=0)
    img = np.expand_dims(img, axis=0)
    return img.astype(np.float32)

# ============================================================
# 10. SOFTMAX FUNCTION
# ============================================================
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# ============================================================
# 11. LOAD ONNX MODELS
# ============================================================
print("\nLoading ONNX models...")
mobilevit_session = ort.InferenceSession(MOBILEVIT_ONNX_PATH)
resnet_session = ort.InferenceSession(RESNET_ONNX_PATH)

mobilevit_input_name = mobilevit_session.get_inputs()[0].name
resnet_input_name = resnet_session.get_inputs()[0].name

# ============================================================
# 12. GENERATE META FEATURES
# ============================================================
print("\nGenerating ensemble meta-features...")
meta_features = []
meta_labels = []

for idx in range(len(features_scaled)):
    row = features_scaled[idx]
    label = labels.iloc[idx]

    img = create_image_from_row(row)

    resnet_input = preprocess_resnet(img)
    mobilevit_input = preprocess_mobilevit(img)

    resnet_output = resnet_session.run(None, {resnet_input_name: resnet_input})[0]
    resnet_probs = softmax(resnet_output)

    mobilevit_output = mobilevit_session.run(None, {mobilevit_input_name: mobilevit_input})[0]
    mobilevit_probs = softmax(mobilevit_output)

    resnet_conf = np.max(resnet_probs[0])
    mobilevit_conf = np.max(mobilevit_probs[0])

    combined_features = np.concatenate([
        resnet_probs[0],
        mobilevit_probs[0],
        np.array([resnet_conf, mobilevit_conf])
    ])

    meta_features.append(combined_features)
    meta_labels.append(label)

X_meta = np.array(meta_features)
y_meta = np.array(meta_labels)

# ============================================================
# 13. LABEL ENCODING
# ============================================================
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_meta)

# ============================================================
# 14. TRAIN / TEST SPLIT
# ============================================================
indices = np.arange(len(X_meta))
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

X_train, X_test = X_meta[train_idx], X_meta[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

# ============================================================
# 15. SMOOTHED WEIGHTS COMPUTATION
# ============================================================
def compute_smoothed_weights(y, smooth_factor=0.15):
    classes, counts = np.unique(y, return_counts=True)
    total_samples = len(y)
    safe_counts = np.where(counts == 0, 1e-6, counts)
    smoothed_weights_array = total_samples / (safe_counts ** smooth_factor)
    smoothed_weights_array /= np.mean(smoothed_weights_array)
    return {cls: weight for cls, weight in zip(classes, smoothed_weights_array)}

class_weight_dict_selected = compute_smoothed_weights(y_train, smooth_factor=0.25)

# ============================================================
# 16. ABLATION STUDY
# ============================================================
print("\n" + "="*60)
print("STARTING ABLATION STUDY RUNS")
print("="*60)

ablation_results = {}

def track_metrics(config_name, y_true, y_pred):
    report = classification_report(y_true, y_pred, target_names=label_encoder.classes_, output_dict=True, zero_division=0)
    ablation_results[config_name] = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Macro)": report["macro avg"]["precision"],
        "Recall (Macro)": report["macro avg"]["recall"],
        "F1-Score (Macro)": report["macro avg"]["f1-score"]
    }

# Slice feature sets cleanly from X_meta configuration mappings
X_train_resnet_probs = X_train[:, :6]
X_test_resnet_probs = X_test[:, :6]

X_train_transformer_probs = X_train[:, 6:12]
X_test_transformer_probs = X_test[:, 6:12]

X_train_raw = features.iloc[train_idx].values
X_test_raw = features.iloc[test_idx].values

# --- 1. Modèle CNN seul ---
print("Evaluating: Modèle CNN seul...")
y_pred_cnn = np.argmax(X_test_resnet_probs, axis=1)
track_metrics("Modèle CNN seul", y_test, y_pred_cnn)

# --- 2. Transformer seul ---
print("Evaluating: Transformer seul...")
y_pred_trans = np.argmax(X_test_transformer_probs, axis=1)
track_metrics("Transformer seul", y_test, y_pred_trans)

# --- 3. RandomForest seul ---
print("Evaluating: RandomForest seul...")
rf_raw = RandomForestClassifier(
    random_state=42, n_jobs=-1, class_weight=class_weight_dict_selected,
    n_estimators=300, min_samples_split=4, min_samples_leaf=3, max_features='log2'
)
rf_raw.fit(X_train_raw, y_train)
y_pred_rf_raw = rf_raw.predict(X_test_raw)
track_metrics("RandomForest seul", y_test, y_pred_rf_raw)

# --- 4. CNN + RF ---
print("Evaluating: CNN + RF...")
rf_cnn = RandomForestClassifier(
    random_state=42, n_jobs=-1, class_weight=class_weight_dict_selected,
    n_estimators=300, min_samples_split=4, min_samples_leaf=3, max_features='log2'
)
rf_cnn.fit(X_train_resnet_probs, y_train)
y_pred_rf_cnn = rf_cnn.predict(X_test_resnet_probs)
track_metrics("CNN + RF", y_test, y_pred_rf_cnn)

# --- 5. Transformer + RF ---
print("Evaluating: Transformer + RF...")
rf_trans = RandomForestClassifier(
    random_state=42, n_jobs=-1, class_weight=class_weight_dict_selected,
    n_estimators=300, min_samples_split=4, min_samples_leaf=3, max_features='log2'
)
rf_trans.fit(X_train_transformer_probs, y_train)
y_pred_rf_trans = rf_trans.predict(X_test_transformer_probs)
track_metrics("Transformer + RF", y_test, y_pred_rf_trans)

# --- 6. Modèle complet ---
print("Evaluating: Modèle complet...")
rf_complete = RandomForestClassifier(
    random_state=42, n_jobs=-1, class_weight=class_weight_dict_selected,
    n_estimators=300, min_samples_split=4, min_samples_leaf=3, max_features='log2'
)
rf_complete.fit(X_train, y_train)
y_pred_complete = rf_complete.predict(X_test)
track_metrics("Modèle complet", y_test, y_pred_complete)

# ============================================================
# 17. DISPLAY FINAL ABLATION RESULTS
# ============================================================
df_ablation = pd.DataFrame(ablation_results).T

print("\n" + "="*85)
print(" FINAL ABLATION STUDY COMPARISON ")
print("="*85)
print(df_ablation.round(4))
print("="*85)

# Save results for reporting
df_ablation.to_csv("ablation_study_results.csv", index_label="Architecture")
print("\nResults exported successfully to 'ablation_study_results.csv'.")

All base required files found.

Loading dataset...
Dataset shape: (1696725, 79)

Class Distribution:
Category
BENIGN      1362791
DoS          227842
Fuzzy         96186
Patator        8299
Advanced       1203
RPM             404
Name: count, dtype: int64

Preparing features...

Subsampling dataset to 3000 rows...
Final features shape for pipeline: (3000, 78)

Loading ONNX models...

Generating ensemble meta-features...

STARTING ABLATION STUDY RUNS
Evaluating: Modèle CNN seul...
Evaluating: Transformer seul...
Evaluating: RandomForest seul...
Evaluating: CNN + RF...
Evaluating: Transformer + RF...
Evaluating: Modèle complet...

 FINAL ABLATION STUDY COMPARISON 
                   Accuracy  Precision (Macro)  Recall (Macro)  \
Modèle CNN seul      0.0100             0.1676          0.1677   
Transformer seul     0.7833             0.1306          0.1667   
RandomForest seul    0.9833             0.9121          0.8935   
CNN + RF             0.9033             0.7729          0.8109   

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. Configuration and Column Name Mapping
# ==========================================
# Change these if your actual columns are named differently
TARGET_COL = 'Label'         # Name of target variable in train_data.csv / Test_data.csv
TRUE_LABEL_COL = 'true_label'     # Name of true label column in prediction CSVs
PROB_COLS = ['cnn_confidence', 'prob_1'] # Columns containing predicted probabilities for each class

# ==========================================
# 2. Load the Tabular Data for Random Forest
# ==========================================
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('Test_data.csv')

X_train = train_data.drop(columns=[TARGET_COL])
y_train = train_data[TARGET_COL]
X_test = test_data.drop(columns=[TARGET_COL])
y_test = test_data[TARGET_COL]

# ==========================================
# 3. Train the "RF seul" (Random Forest Only)
# ==========================================
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Get RF probabilities on Test Set
rf_probs = rf.predict_proba(X_test)

# ==========================================
# 4. Load Deep Learning Predictions (Test Set)
# ==========================================
# Assuming the rows in the tabular data map exactly 1:1 to the rows in the prediction CSVs
resnet_test = pd.read_csv('prob_files/resnet_test_predictions.csv')
mobilvit_test = pd.read_csv('prob_files/mobilvit_test_predictions.csv')

cnn_probs = resnet_test[PROB_COLS].values
transformer_probs = mobilvit_test[PROB_COLS].values
y_true = resnet_test[TRUE_LABEL_COL].values # Ground truth from Deep Learning test set

# ==========================================
# 5. Define the Ablation Study Combinations
# ==========================================
# We use soft voting (averaging the probabilities) to combine models
ablation_results = {}
ablation_predictions = {}

models = {
    "Modèle CNN seul": cnn_probs,
    "Transformer seul": transformer_probs,
    "RF seul": rf_probs,
    "CNN + RF": (cnn_probs + rf_probs) / 2,
    "Transformer + RF": (transformer_probs + rf_probs) / 2,
    "Modèle complet (CNN + Transf. + RF)": (cnn_probs + transformer_probs + rf_probs) / 3
}

# Evaluate Each Model
for model_name, probs in models.items():
    # Convert probabilities to class predictions
    y_pred = np.argmax(probs, axis=1)
    ablation_predictions[model_name] = y_pred
    
    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    ablation_results[model_name] = {'Accuracy': acc, 'F1-Score': f1}

# Display the Results
print("=== Ablation Study Results ===")
results_df = pd.DataFrame(ablation_results).T
print(results_df.round(4))
print("\n")

# ==========================================
# 6. Plot Normalized Confusion Matrices
# ==========================================
def plot_normalized_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, normalize='true') # Normalize over true (rows)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', 
                cbar=True, annot_kws={"size": 12})
    
    plt.title(title, fontsize=14)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()

# Plot for the Complete Model
plot_normalized_cm(y_true, ablation_predictions["Modèle complet (CNN + Transf. + RF)"], 
                   "Normalized Confusion Matrix: Modèle Complet")

# Optional: If you want to see all matrices, simply loop through them:
# for name, y_pred in ablation_predictions.items():
#     plot_normalized_cm(y_true, y_pred, f"Normalized CM: {name}")

In [5]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ==========================================
# 1. Label Mapping & Class Configuration
# ==========================================
CLASSES = ["BENIGN", "DoS", "Fuzzy", "RPM", "Advanced", "Patator"]

def map_label(label):
    if not isinstance(label, str):
        return label
    if label == "BENIGN":
        return "BENIGN"
    elif "DoS" in label:
        return "DoS"
    elif "PortScan" in label or "Brute Force" in label:
        return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label:
        return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label:
        return "Advanced"
    elif "Patator" in label:
        return "Patator"
    return label

# Define your custom column naming logic
# NOTE: If your CSVs don't use underscores (e.g., 'cnnBENIGN'), change f"cnn_{c}" to f"cnn{c}"
CNN_COLS = [f"cnn_{c}" for c in CLASSES] + ["cnn_confidence"]
TRANS_COLS = [f"trans_{c}" for c in CLASSES] + ["trans_confidence"]

# CSV Configuration
 
TABULAR_LABEL_COL = 'Label'  # Name of target column in train_data/Test_data
DL_TRUE_LABEL_COL = 'true_label' # Name of true label column in prediction CSVs

# ==========================================
# 2. Load & Clean Tabular Data (RF)
# ==========================================
print("Loading and mapping tabular data...")
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('Test_data.csv')

# Unify labels using your mapping function
train_data[TABULAR_LABEL_COL] = train_data[TABULAR_LABEL_COL].apply(map_label)
test_data[TABULAR_LABEL_COL] = test_data[TABULAR_LABEL_COL].apply(map_label)

X_train = train_data.drop(columns=[TABULAR_LABEL_COL])
y_train = train_data[TABULAR_LABEL_COL]
X_test = test_data.drop(columns=[TABULAR_LABEL_COL])
y_test = test_data[TABULAR_LABEL_COL]

# ==========================================
# 3. Train RF & Align Probability Output
# ==========================================
print("Training Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Scikit-learn outputs probabilities alphabetically based on rf.classes_
rf_probs_raw = rf.predict_proba(X_test)

# Crucial step: Reorder RF probabilities to match the exact order of CLASSES array
rf_probs = np.zeros((len(X_test), len(CLASSES)))
for i, cls in enumerate(CLASSES):
    class_idx = np.where(rf.classes_ == cls)[0][0]
    rf_probs[:, i] = rf_probs_raw[:, class_idx]

# ==========================================
# 4. Load Deep Learning Predictions
# ==========================================
print("Loading CNN and Transformer predictions...")
resnet_test = pd.read_csv('prob_files/resnet_val_predictions_new.csv')
mobilvit_test = pd.read_csv('prob_files/mobilvit_val_predictions.csv')

# Extracted probabilities directly matching our class order
cnn_probs = resnet_test[CNN_COLS].values
transformer_probs = mobilvit_test[TRANS_COLS].values

# Map the true labels in the DL test files just in case they aren't mapped yet
y_true = resnet_test[DL_TRUE_LABEL_COL].apply(map_label).values

# Convert string true labels to their matching integer indices for metric evaluations
y_true_idx = np.array([CLASSES.index(label) for label in y_true])

# ==========================================
# 5. Execute Ablation Study
# ==========================================
models = {
    "Modèle CNN seul": cnn_probs,
    "Transformer seul": transformer_probs,
    "RF seul": rf_probs,
    "CNN + RF": (cnn_probs + rf_probs) / 2,
    "Transformer + RF": (transformer_probs + rf_probs) / 2,
    
}

ablation_results = {}
ablation_predictions = {}

for name, probs in models.items():
    y_pred_idx = np.argmax(probs, axis=1)
    ablation_predictions[name] = y_pred_idx
    
    acc = accuracy_score(y_true_idx, y_pred_idx)
    f1 = f1_score(y_true_idx, y_pred_idx, average='weighted')
    
    ablation_results[name] = {'Accuracy': acc, 'F1-Score': f1}

# Display results table
print("\n=== ABLATION STUDY RESULTS ===")
results_df = pd.DataFrame(ablation_results).T
print(results_df.round(4))
print("==============================\n")

# ==========================================
# 6. Normalized Confusion Matrix (Modèle complet)
# ==========================================
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred, normalize='true') # Normalize by rows (True labels)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                cbar=True, annot_kws={"size": 11})
    
    plt.title(title, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Generate the plot for the complete ensemble model
plot_confusion_matrix(
    y_true_idx, 
    ablation_predictions["Modèle complet"], 
    CLASSES, 
    "Normalized Confusion Matrix: Modèle Complet"
)

Loading and mapping tabular data...
Training Random Forest Classifier...
Loading CNN and Transformer predictions...


ValueError: operands could not be broadcast together with shapes (565575,7) (565576,6) 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ==========================================
# 1. Configuration & Label Mapping
# ==========================================
CLASSES = ["BENIGN", "DoS", "Fuzzy", "RPM", "Advanced", "Patator"]

def map_label(label):
    if not isinstance(label, str):
        return label
    if label == "BENIGN":
        return "BENIGN"
    elif "DoS" in label:
        return "DoS"
    elif "PortScan" in label or "Brute Force" in label:
        return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label:
        return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label:
        return "Advanced"
    elif "Patator" in label:
        return "Patator"
    return label

# Setup Column Names
CNN_COLS = [f"cnn_{c}" for c in CLASSES] + ["cnn_confidence"]
TRANS_COLS = [f"trans_{c}" for c in CLASSES] + ["trans_confidence"]


TABULAR_LABEL_COL = 'Label'  # Name of target column in train_data/Test_data
DL_TRUE_LABEL_COL = 'true_label' 
# ==========================================
# 2. Load & Prepare Tabular Data (RF Seul)
# ==========================================
print("Processing Tabular Data...")
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('Test_data.csv')

train_data[TABULAR_LABEL_COL] = train_data[TABULAR_LABEL_COL].apply(map_label)
test_data[TABULAR_LABEL_COL] = test_data[TABULAR_LABEL_COL].apply(map_label)

X_train_tab = train_data.drop(columns=[TABULAR_LABEL_COL])
X_test_tab = test_data.drop(columns=[TABULAR_LABEL_COL])

# Map targets to exact indices (0 to 5) to keep everything unified
y_train_idx = np.array([CLASSES.index(l) for l in train_data[TABULAR_LABEL_COL]])
y_test_idx = np.array([CLASSES.index(l) for l in test_data[TABULAR_LABEL_COL]])

# Train Base Tabular RF
rf_tabular = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_tabular.fit(X_train_tab, y_train_idx)

# Generate probabilities for Stacking
rf_tab_train_probs = rf_tabular.predict_proba(X_train_tab)
rf_tab_test_probs = rf_tabular.predict_proba(X_test_tab)

# ==========================================
# 3. Load & Fix Deep Learning Predictions
# ==========================================
print("Loading and fixing Deep Learning Prediction files...")

# Load ResNet (CNN)
resnet_train = pd.read_csv('prob_files/resnet_train_predictions_new.csv')
resnet_test = pd.read_csv('prob_files/resnet_test_predictions.csv')

# Load MobileViT (Transformer) & FIX the 'cnn_' to 'trans_' typo on the fly
mobilvit_train = pd.read_csv('prob_files/mobilvit_train_predictions.csv').rename(columns=lambda x: x.replace('cnn_', 'trans_'))
mobilvit_test = pd.read_csv('prob_files/mobilvit_test_predictions.csv').rename(columns=lambda x: x.replace('cnn_', 'trans_'))

# Extract pure probability features (excluding confidence columns)
cnn_train_feats = resnet_train[CNN_COLS].values
cnn_test_feats = resnet_test[CNN_COLS].values

trans_train_feats = mobilvit_train[TRANS_COLS].values
trans_test_feats = mobilvit_test[TRANS_COLS].values

# ==========================================
# 4. Train Stacking Meta-Classifiers (Passing Probs Through RF)
# ==========================================
print("Training Stacking Meta-Classifiers...")

# --- CNN + RF ---
rf_cnn_stack = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_cnn_stack.fit(cnn_train_feats, y_train_idx)
preds_cnn_rf = rf_cnn_stack.predict(cnn_test_feats)

# --- Transformer + RF ---
rf_trans_stack = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_trans_stack.fit(trans_train_feats, y_train_idx)
preds_trans_rf = rf_trans_stack.predict(trans_test_feats)

# --- Modèle Complet (CNN Probs + Trans Probs + Tabular RF Probs stacked together) ---
X_train_complet = np.hstack((cnn_train_feats, trans_train_feats, rf_tab_train_probs))
X_test_complet = np.hstack((cnn_test_feats, trans_test_feats, rf_tab_test_probs))

rf_complet_stack = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_complet_stack.fit(X_train_complet, y_train_idx)
preds_mod_complet = rf_complet_stack.predict(X_test_complet)

# ==========================================
# 5. Evaluate All Models (Ablation Study)
# ==========================================
# Base standalone predictions (argmax of raw test probabilities)
preds_cnn_seul = np.argmax(cnn_test_feats, axis=1)
preds_trans_seul = np.argmax(trans_test_feats, axis=1)
preds_rf_seul = rf_tabular.predict(X_test_tab)

ablation_predictions = {
    "Modèle CNN seul": preds_cnn_seul,
    "Transformer seul": preds_trans_seul,
    "RF seul": preds_rf_seul,
    "CNN + RF (Stacked)": preds_cnn_rf,
    "Transformer + RF (Stacked)": preds_trans_rf,
    "Modèle complet (Stacked)": preds_mod_complet
}

ablation_results = {}
for name, y_pred in ablation_predictions.items():
    acc = accuracy_score(y_test_idx, y_pred)
    f1 = f1_score(y_test_idx, y_pred, average='weighted')
    ablation_results[name] = {'Accuracy': acc, 'F1-Score': f1}

# Print Table
print("\n=== ABLATION STUDY RESULTS (STACKING ARCHITECTURE) ===")
results_df = pd.DataFrame(ablation_results).T
print(results_df.round(4))
print("======================================================\n")

# ==========================================
# 6. Plot Normalized Confusion Matrix
# ==========================================
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred, normalize='true') # Row-normalized (Recall)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                cbar=True, annot_kws={"size": 11})
    
    plt.title(title, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Generate plot for the stacked complete model
plot_confusion_matrix(
    y_test_idx, 
    ablation_predictions["Modèle complet (Stacked)"], 
    CLASSES, 
    "Normalized Confusion Matrix: Modèle Complet (Stacked)"
)

Processing Tabular Data...
